# CIF Density Calculator

Theoretical density from crystallographic CIF files - batch processing,
built-in validation, cross-platform.

This notebook is the user interface; the calculation lives in
[`cif_density.py`](cif_density.py) and the full validation suite in
[`test_cif_density.py`](test_cif_density.py) (run with `pytest`).
Full documentation: see [`README.md`](README.md).

## 1. Setup

Installs any missing dependency, then imports the calculation engine from
`cif_density.py` - keep that file next to the notebook (on Google Colab:
upload both files together).

In [ ]:
# Dependency bootstrap: installs pymatgen/pandas only if missing. Plain
# Python (no notebook magics), so it also works headless (nbconvert/CI).
import importlib.util
import subprocess
import sys
from pathlib import Path

for _pkg in ("pymatgen", "pandas"):
    if importlib.util.find_spec(_pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", _pkg])
        print(f"Installed {_pkg}")

if not Path("cif_density.py").is_file():
    raise FileNotFoundError(
        "cif_density.py not found next to the notebook - upload/copy it "
        "into the working directory (on Colab: Files sidebar).")
# Imported only here: the bootstrap above must run first, or a missing
# pymatgen would raise ModuleNotFoundError before it can be installed.
# display is imported explicitly rather than relying on the name IPython
# injects, so the cells still run once exported with nbconvert --to script.
from IPython.display import display

from cif_density import __version__, process_folder, render_results

print(f"cif_density {__version__} loaded")

## 2. Quick self-check (optional)

One synthetic NaCl structure is generated in a temporary directory and its
density compared against the hand-calculated reference - a fast end-to-end
check that the environment works. The full validation suite (six synthetic
cases, two independent tolerance levels) lives in `test_cif_density.py`:

```bash
pytest
```

Run it without a path: `pyproject.toml` sets `--doctest-modules`, so a bare
`pytest` also executes the `>>>` examples in `cif_density.py`. Naming the
test file explicitly skips them.

In [ ]:
import tempfile

# NaCl rock salt, a = 5.6402 A; reference density from IUPAC atomic weights.
_NACL_CIF = """data_NaCl
_cell_length_a 5.6402
_cell_length_b 5.6402
_cell_length_c 5.6402
_cell_angle_alpha 90
_cell_angle_beta 90
_cell_angle_gamma 90
_symmetry_space_group_name_H-M 'F m -3 m'
loop_
_atom_site_label
_atom_site_type_symbol
_atom_site_fract_x
_atom_site_fract_y
_atom_site_fract_z
_atom_site_occupancy
Na1 Na 0 0 0 1
Cl1 Cl 0.5 0.5 0.5 1
"""
_RHO_REF = 2.164  # g/cm^3

with tempfile.TemporaryDirectory() as _tmp:
    Path(_tmp, "NaCl.cif").write_text(_NACL_CIF)
    _results, _errors = process_folder(_tmp)

_rho = _results["Density (g/cm^3)"].iloc[0]
assert _errors.empty and abs(_rho - _RHO_REF) / _RHO_REF < 2e-3, (
    f"self-check failed: rho = {_rho}")
print(f"Self-check passed: NaCl density = {_rho:.4f} g/cm^3 "
      f"(reference {_RHO_REF})")

## 3. Analyze your own CIF files

Copy your `.cif` files into `cif_files/` (create it next to this notebook,
or in the Colab Files sidebar), adjust the parameters below if needed and
run. Results are shown as a table and written to `density_results.csv`.
The error table below it lists anything that did not make it into the
results: unreadable files, and phases the parser could not build. Neither
stops the batch, but both mean a row is missing, so it is worth reading
when it appears. Step-by-step instructions are in the README.

In [ ]:
INPUT_FOLDER = "cif_files"           # folder containing your .cif files
OUTPUT_CSV = "density_results.csv"   # written next to this notebook

folder = Path(INPUT_FOLDER)
folder.mkdir(exist_ok=True)  # first run: created empty, ready for your files
results, errors = process_folder(folder, OUTPUT_CSV)
if results.empty:
    print(f"No .cif files in '{INPUT_FOLDER}': add your files and re-run this cell.")
else:
    display(render_results(results, caption=f"{len(results)} phase(s) -> {OUTPUT_CSV}"))
if not errors.empty:
    # Unreadable files and phases the parser could not build both land here.
    print("\nNot included in the results:")
    print(errors.to_string(index=False))